# Coconut Latent-CoT on LFM2.5-350M -- Production Build

This notebook takes the research prototype (continuous-thought / Coconut wrapper around
`LiquidAI/LFM2.5-350M`) and turns it into something you can actually ship. It keeps the
core idea -- replacing verbose chain-of-thought tokens with `K` continuous hidden-state
"thoughts" fed back as embeddings -- but fixes the parts that make the original unusable
in production, and applies it to **three real workloads from your own stack**:

1. **AVA (WhatsApp sales agent) -- escalate-to-human gate**
2. **Intraday signal fusion** (RSI / MACD / volume -> BUY/SELL/HOLD)
3. **On-device deployment** for the Android/Expo app

## What was wrong with the research version (and what's fixed here)

| Issue in the research notebook | Fix in this notebook |
|---|---|
| Latent loop re-runs the full growing sequence every step (`use_cache=False`), O(n^2) cost | Latent loop uses `past_key_values` / KV-cache, O(n) |
| Token-by-token generation also recomputes the whole sequence every new token | Generation reuses the cache from the latent phase; only 1 new token per step is forwarded |
| Trained on a synthetic `op(a,b,c)` arithmetic task | Liquid AI's own release notes for LFM2.5-350M state it is "not recommended for tasks like math, code, and creative writing" and is tuned instead for tool use, structured output and instruction following [web:14]. Arithmetic is the worst possible fit for this model. All three product scenarios below are reframed as classification / structured-decision tasks instead. |
| No train/val split, no checkpointing, no early stopping | Added |
| No inference wrapper with error handling / logging / timeouts | Added |
| No latency/memory benchmarking | Added |
| No deployment path | Added, with a grounded caveat about why the raw PyTorch latent loop can't be dropped onto llama.cpp/GGUF/ONNX Mobile as-is |
| Synthetic data used as if it were real | Every dataset generator below is labeled PLACEHOLDER where that applies |

> Data honesty note: I have no access to your actual AVA conversation logs, WhatsApp
> webhook payloads, or trading indicator history. The dataset generators in Scenarios 1
> and 2 are schema demonstrations with clearly-labeled synthetic values -- swap in real
> logged data before training anything you intend to trust. This notebook was not
> executed end-to-end here (no GPU/internet/torch in this sandbox), so there are no
> fabricated metrics below -- every number you see after running it will be real.


## Part 0 -- Environment & Setup

In [ ]:
# Core deps. bitsandbytes is optional (Linux/NVIDIA only) for 8-bit loading.
!pip install -q torch transformers accelerate huggingface_hub onnx onnxruntime


In [ ]:
import os
import time
import json
import logging
from typing import Optional, Dict, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("coconut_prod")

torch.manual_seed(42)

if torch.cuda.is_available():
    device, dtype = torch.device("cuda"), torch.bfloat16
elif torch.backends.mps.is_available():
    device, dtype = torch.device("mps"), torch.float32
else:
    device, dtype = torch.device("cpu"), torch.float32

log.info(f"Using device={device}, dtype={dtype}")

MODEL_ID = "LiquidAI/LFM2.5-350M"
HF_TOKEN = os.environ.get("HF_TOKEN")  # set this to avoid HF rate limiting

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=dtype, trust_remote_code=True, token=HF_TOKEN
).to(device)
base_model.eval()

hidden_dim = base_model.config.hidden_size
vocab_size = base_model.config.vocab_size
log.info(f"Loaded {MODEL_ID}: hidden_dim={hidden_dim}, params={base_model.num_parameters():,}")


## Part 1 -- Production Coconut Wrapper

Key differences from the research version:
- KV-cache reuse in both the latent-thought loop and generation (no O(n^2) recompute).
- Bounded, validated inputs (raises instead of silently crashing on empty input).
- Deterministic-by-default generation (temperature=0.0) because all three product
  scenarios need a stable, auditable decision, not creative sampling.
- Structured dict output instead of a bare string, so a FastAPI endpoint / Celery task /
  the Expo app (via REST) can consume it directly.


In [ ]:
class ProductionCoconut(nn.Module):
    """Latent chain-of-thought wrapper with KV-cache-based recurrence.

    num_latent_steps: how many continuous 'thought' vectors to inject before decoding.
    Keep this small (2-4) for a 350M model -- more steps does not reliably help and
    costs latency. Tune empirically per scenario on a held-out validation set.
    """

    def __init__(self, model, num_latent_steps: int = 3):
        super().__init__()
        self.model = model
        self.num_latent_steps = num_latent_steps
        self.embed_tokens = model.get_input_embeddings()
        self.hidden_dim = model.config.hidden_size
        self.latent_proj = nn.Linear(
            self.hidden_dim, self.hidden_dim, bias=False,
            dtype=model.dtype, device=model.device,
        )
        nn.init.eye_(self.latent_proj.weight)

    def _run_latent_phase(self, input_ids: torch.Tensor):
        """Prefill the prompt once, then unroll K latent thoughts using cached KV.
        Returns (past_key_values, last_latent_embed) ready for the decode phase.
        """
        embeds = self.embed_tokens(input_ids)
        out = self.model(inputs_embeds=embeds, output_hidden_states=True, use_cache=True)
        past = out.past_key_values
        last_hidden = out.hidden_states[-1][:, -1:, :]

        for _ in range(self.num_latent_steps):
            latent_vec = self.latent_proj(last_hidden)
            out = self.model(inputs_embeds=latent_vec, past_key_values=past,
                              output_hidden_states=True, use_cache=True)
            past = out.past_key_values
            last_hidden = out.hidden_states[-1][:, -1:, :]

        return past, self.latent_proj(last_hidden)

    def forward(self, input_ids: torch.Tensor, target_ids: Optional[torch.Tensor] = None):
        if input_ids.numel() == 0:
            raise ValueError("input_ids must be non-empty")

        past, last_latent = self._run_latent_phase(input_ids)

        if target_ids is None:
            return past, last_latent

        target_embeds = self.embed_tokens(target_ids)
        decode_embeds = torch.cat([last_latent, target_embeds], dim=1)
        out = self.model(inputs_embeds=decode_embeds, past_key_values=past, use_cache=False)
        logits = out.logits

        tgt_len = target_ids.shape[1]
        pred_logits = logits[:, -(tgt_len + 1):-1, :].contiguous()
        loss = F.cross_entropy(
            pred_logits.reshape(-1, self.model.config.vocab_size),
            target_ids.reshape(-1),
            ignore_index=tokenizer.pad_token_id,
        )
        return loss, pred_logits

    @torch.no_grad()
    def generate(self, input_ids: torch.Tensor, max_new_tokens: int = 16,
                 temperature: float = 0.0) -> Dict[str, Any]:
        """Deterministic-by-default decode. Returns text + timing, never raises on
        failure (returns an 'error' key instead) so callers can apply their own fallback.
        """
        self.eval()
        t0 = time.perf_counter()
        try:
            past, cur_embed = self.forward(input_ids)
        except Exception as e:
            log.exception("Latent phase failed")
            return {"text": "", "tokens": [], "latency_ms": None, "error": str(e)}

        generated = []
        for _ in range(max_new_tokens):
            out = self.model(inputs_embeds=cur_embed, past_key_values=past, use_cache=True)
            past = out.past_key_values
            next_logits = out.logits[:, -1, :]

            if temperature == 0.0:
                next_token = torch.argmax(next_logits, dim=-1, keepdim=True)
            else:
                probs = F.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)

            tok_id = next_token.item()
            if tok_id == tokenizer.eos_token_id:
                break
            generated.append(tok_id)
            cur_embed = self.embed_tokens(next_token)

        latency_ms = (time.perf_counter() - t0) * 1000
        text = tokenizer.decode(generated, skip_special_tokens=True)
        return {"text": text.strip(), "tokens": generated, "latency_ms": round(latency_ms, 2)}


## Part 2 -- Scenario 1: AVA WhatsApp Escalation Gate

**Real usage**: inside AVA (your WhatsApp AI sales agent), every inbound customer message
already produces signals your pipeline computes anyway -- lead score, sentiment, number of
price objections, message count in the thread, deal size tier. Today that logic is
probably a hand-written if/else chain. Here we replace the *decision* step (not the whole
conversation!) with a tiny latent-reasoning classifier: given the structured signals, output
one token: `ESCALATE` or `AUTO`. This is exactly the kind of structured-output /
classification task LFM2.5-350M is tuned for [web:14], and it runs in a few milliseconds on
CPU, cheap enough to call on every inbound message.

> PLACEHOLDER DATA: `generate_ava_training_row` below encodes a *schema*, not real
> conversations. Before training on this for real, export your actual AVA event logs with
> the same five fields plus the human agent's real outcome (did a human actually take over
> that chat?) as the label. Do not deploy a model trained only on synthetic rows.


In [ ]:
import random

def generate_ava_training_row(rng: random.Random):
    """PLACEHOLDER schema-only generator. Replace with a query against your real
    AVA conversation-log store (fields below should already exist in your pipeline).
    """
    message_count = rng.randint(1, 20)
    sentiment = round(rng.uniform(-1.0, 1.0), 2)          # -1 very negative .. 1 very positive
    price_objections = rng.randint(0, 4)
    deal_size_tier = rng.randint(1, 5)                     # 1 = small, 5 = enterprise
    negative_flag = rng.choice([0, 1])

    # Bootstrap rule -- REPLACE with real "did a human actually take over" labels.
    risk_score = (
        (message_count > 10) * 1
        + (sentiment < -0.2) * 2
        + (price_objections >= 2) * 2
        + (deal_size_tier >= 4) * 1
        + negative_flag * 1
    )
    label = "ESCALATE" if risk_score >= 3 else "AUTO"

    prompt = (
        f"Lead signals: messages={message_count}, sentiment={sentiment}, "
        f"price_objections={price_objections}, deal_tier={deal_size_tier}, "
        f"negative_flag={negative_flag}.\nDecision:"
    )
    return prompt, f" {label}"


class AVAEscalationDataset(Dataset):
    def __init__(self, tokenizer, n=600, seed=0):
        rng = random.Random(seed)
        self.data = []
        for _ in range(n):
            prompt, label = generate_ava_training_row(rng)
            p_ids = tokenizer.encode(prompt, add_special_tokens=False)
            a_ids = tokenizer.encode(label, add_special_tokens=False) + [tokenizer.eos_token_id]
            self.data.append((torch.tensor(p_ids), torch.tensor(a_ids)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def collate_fn(batch):
    prompts, targets = zip(*batch)
    p = torch.nn.utils.rnn.pad_sequence(prompts, batch_first=True, padding_value=tokenizer.pad_token_id)
    t = torch.nn.utils.rnn.pad_sequence(targets, batch_first=True, padding_value=tokenizer.pad_token_id)
    return p, t


full_ds = AVAEscalationDataset(tokenizer, n=600, seed=0)
n_val = int(0.15 * len(full_ds))
train_ds, val_ds = torch.utils.data.random_split(
    full_ds, [len(full_ds) - n_val, n_val], generator=torch.Generator().manual_seed(42)
)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, collate_fn=collate_fn)
log.info(f"AVA dataset: {len(train_ds)} train / {len(val_ds)} val rows")


In [ ]:
ava_model = ProductionCoconut(base_model, num_latent_steps=3)

for p in ava_model.model.parameters():
    p.requires_grad = False
for p in ava_model.latent_proj.parameters():
    p.requires_grad = True
if hasattr(ava_model.model, "model") and hasattr(ava_model.model.model, "layers"):
    for layer in ava_model.model.model.layers[-2:]:
        for p in layer.parameters():
            p.requires_grad = True

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, ava_model.parameters()), lr=2e-4, weight_decay=0.01
)

best_val_loss = float("inf")
patience, patience_ctr = 3, 0
CKPT_PATH = "ava_escalation_gate.pt"

for epoch in range(10):
    ava_model.train()
    train_loss = 0.0
    for prompts, targets in train_loader:
        prompts, targets = prompts.to(device), targets.to(device)
        optimizer.zero_grad()
        loss, _ = ava_model(prompts, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ava_model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    ava_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for prompts, targets in val_loader:
            prompts, targets = prompts.to(device), targets.to(device)
            loss, _ = ava_model(prompts, targets)
            val_loss += loss.item()
    val_loss /= len(val_loader)

    log.info(f"Epoch {epoch+1}: train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_ctr = 0
        torch.save({
            "latent_proj": ava_model.latent_proj.state_dict(),
            "num_latent_steps": ava_model.num_latent_steps,
            "val_loss": val_loss,
        }, CKPT_PATH)
        log.info(f"  saved checkpoint (val_loss={val_loss:.4f})")
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            log.info("Early stopping triggered")
            break


In [ ]:
class AVAEscalationGate:
    """Production wrapper: call .decide(features) from your webhook handler
    (e.g. the FastAPI/Express service in front of AVA) on every inbound message.
    """

    def __init__(self, coconut_model: ProductionCoconut, checkpoint_path: str):
        ckpt = torch.load(checkpoint_path, map_location=device)
        coconut_model.latent_proj.load_state_dict(ckpt["latent_proj"])
        coconut_model.num_latent_steps = ckpt["num_latent_steps"]
        self.model = coconut_model
        self.model.eval()

    def decide(self, message_count: int, sentiment: float, price_objections: int,
               deal_size_tier: int, negative_flag: int) -> Dict[str, Any]:
        prompt = (
            f"Lead signals: messages={message_count}, sentiment={sentiment}, "
            f"price_objections={price_objections}, deal_tier={deal_size_tier}, "
            f"negative_flag={negative_flag}.\nDecision:"
        )
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
        result = self.model.generate(input_ids, max_new_tokens=4, temperature=0.0)

        action = "ESCALATE" if "ESCALATE" in result["text"].upper() else "AUTO"
        result["action"] = action
        # Fail-safe default: on any error, escalate to a human rather than silently
        # auto-replying to a lead you are not confident about.
        if result.get("error"):
            result["action"] = "ESCALATE"
        return result


# Example integration point (pseudo, adapt to your actual webhook framework):
#
# @app.post("/whatsapp/webhook")
# def handle_inbound(payload: dict):
#     features = compute_lead_features(payload)   # your existing pipeline
#     decision = gate.decide(**features)
#     if decision["action"] == "ESCALATE":
#         notify_human_agent(payload["thread_id"])
#     else:
#         ava_auto_reply(payload["thread_id"])

gate = AVAEscalationGate(ava_model, CKPT_PATH)
sample = gate.decide(message_count=14, sentiment=-0.6, price_objections=3, deal_size_tier=5, negative_flag=1)
log.info(f"Sample decision: {sample}")


## Part 3 -- Scenario 2: Intraday Signal Fusion (BUY/SELL/HOLD)

**Real usage**: your intraday tooling already computes indicator deltas per tick
(RSI-50 offset, MACD histogram, volume z-score). Instead of hand-tuned thresholds, fuse
them into a single deterministic signal with the same latent-reasoning gate, so it runs
in milliseconds inside a live tick loop.

> Financial-risk disclaimer, non-negotiable: the label generator below is a **rule-based
> bootstrap**, not a backtested or profitable strategy. Do not connect this to a live
> broker or real capital. Before any real use: (1) replace the rule-based labels with
> outcomes from actual historical bars (e.g. forward N-bar return exceeding a threshold),
> (2) run a proper walk-forward backtest with transaction costs and slippage, (3) paper
> trade before risking capital. This section only demonstrates the *engineering pattern*
> for fusing numeric indicators fast on-device, not a trading edge.


In [ ]:
def generate_trading_row(rng: random.Random):
    """PLACEHOLDER: replace with real indicator values computed from your OHLCV feed."""
    rsi_offset = round(rng.uniform(-50, 50), 1)     # RSI - 50, so 0 = neutral
    macd_hist = round(rng.uniform(-2.0, 2.0), 2)
    vol_zscore = round(rng.uniform(-3.0, 3.0), 2)

    score = rsi_offset * 0.4 + macd_hist * 20 + vol_zscore * 5   # bootstrap rule only
    if score > 15:
        label = "BUY"
    elif score < -15:
        label = "SELL"
    else:
        label = "HOLD"

    prompt = f"Indicators: rsi_offset={rsi_offset}, macd_hist={macd_hist}, vol_z={vol_zscore}.\nSignal:"
    return prompt, f" {label}"


class TradingSignalDataset(Dataset):
    def __init__(self, tokenizer, n=900, seed=1):
        rng = random.Random(seed)
        self.data = []
        for _ in range(n):
            prompt, label = generate_trading_row(rng)
            p_ids = tokenizer.encode(prompt, add_special_tokens=False)
            a_ids = tokenizer.encode(label, add_special_tokens=False) + [tokenizer.eos_token_id]
            self.data.append((torch.tensor(p_ids), torch.tensor(a_ids)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


trading_full = TradingSignalDataset(tokenizer, n=900, seed=1)
n_val = int(0.15 * len(trading_full))
trading_train, trading_val = torch.utils.data.random_split(
    trading_full, [len(trading_full) - n_val, n_val], generator=torch.Generator().manual_seed(42)
)
trading_train_loader = DataLoader(trading_train, batch_size=16, shuffle=True, collate_fn=collate_fn)
trading_val_loader = DataLoader(trading_val, batch_size=16, shuffle=False, collate_fn=collate_fn)
log.info(f"Trading dataset: {len(trading_train)} train / {len(trading_val)} val rows")


In [ ]:
trading_model = ProductionCoconut(base_model, num_latent_steps=2)  # fewer steps: latency-critical

for p in trading_model.model.parameters():
    p.requires_grad = False
for p in trading_model.latent_proj.parameters():
    p.requires_grad = True
if hasattr(trading_model.model, "model") and hasattr(trading_model.model.model, "layers"):
    for layer in trading_model.model.model.layers[-1:]:
        for p in layer.parameters():
            p.requires_grad = True

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, trading_model.parameters()), lr=2e-4, weight_decay=0.01
)

best_val_loss = float("inf")
patience, patience_ctr = 3, 0
TRADING_CKPT = "trading_signal_fusion.pt"

for epoch in range(10):
    trading_model.train()
    train_loss = 0.0
    for prompts, targets in trading_train_loader:
        prompts, targets = prompts.to(device), targets.to(device)
        optimizer.zero_grad()
        loss, _ = trading_model(prompts, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trading_model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(trading_train_loader)

    trading_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for prompts, targets in trading_val_loader:
            prompts, targets = prompts.to(device), targets.to(device)
            loss, _ = trading_model(prompts, targets)
            val_loss += loss.item()
    val_loss /= len(trading_val_loader)

    log.info(f"Epoch {epoch+1}: train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_ctr = 0
        torch.save({
            "latent_proj": trading_model.latent_proj.state_dict(),
            "num_latent_steps": trading_model.num_latent_steps,
            "val_loss": val_loss,
        }, TRADING_CKPT)
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            log.info("Early stopping triggered")
            break


In [ ]:
class TradingSignalFusion:
    """Call .fuse(rsi_offset, macd_hist, vol_zscore) from your tick-processing loop.
    Designed to run in single-digit milliseconds on CPU -- benchmark on your target
    hardware before wiring into a live loop (see Part 4 for the benchmark harness).
    """

    def __init__(self, coconut_model: ProductionCoconut, checkpoint_path: str):
        ckpt = torch.load(checkpoint_path, map_location=device)
        coconut_model.latent_proj.load_state_dict(ckpt["latent_proj"])
        coconut_model.num_latent_steps = ckpt["num_latent_steps"]
        self.model = coconut_model
        self.model.eval()

    def fuse(self, rsi_offset: float, macd_hist: float, vol_zscore: float) -> Dict[str, Any]:
        prompt = f"Indicators: rsi_offset={rsi_offset}, macd_hist={macd_hist}, vol_z={vol_zscore}.\nSignal:"
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
        result = self.model.generate(input_ids, max_new_tokens=3, temperature=0.0)

        text = result["text"].upper()
        signal = "HOLD"
        for candidate in ("BUY", "SELL", "HOLD"):
            if candidate in text:
                signal = candidate
                break
        # Fail-safe: any error or unparsable output defaults to HOLD, never a directional trade.
        if result.get("error"):
            signal = "HOLD"
        result["signal"] = signal
        return result


fusion = TradingSignalFusion(trading_model, TRADING_CKPT)
sample_signal = fusion.fuse(rsi_offset=22.0, macd_hist=0.8, vol_zscore=1.4)
log.info(f"Sample trading signal: {sample_signal}")


## Part 4 -- Latency & Memory Benchmark

Both scenarios above need to be fast (a live WhatsApp reply, a live tick loop). Run this
on the actual hardware you plan to deploy to (your dev machine, the server behind AVA, or
a cloud CPU instance) -- do not trust numbers from a different machine.


In [ ]:
import statistics

def benchmark_wrapper(wrapper_fn, n_runs=30, warmup=5):
    for _ in range(warmup):
        wrapper_fn()
    latencies = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        wrapper_fn()
        latencies.append((time.perf_counter() - t0) * 1000)
    return {
        "p50_ms": round(statistics.median(latencies), 2),
        "p95_ms": round(sorted(latencies)[int(0.95 * len(latencies))], 2),
        "mean_ms": round(statistics.mean(latencies), 2),
    }

ava_bench = benchmark_wrapper(lambda: gate.decide(5, 0.1, 1, 2, 0))
trading_bench = benchmark_wrapper(lambda: fusion.fuse(10.0, 0.3, 0.5))

log.info(f"AVA escalation gate latency: {ava_bench}")
log.info(f"Trading signal fusion latency: {trading_bench}")

param_count = sum(p.numel() for p in base_model.parameters())
approx_fp32_mb = param_count * 4 / (1024 ** 2)
log.info(f"Base model: {param_count:,} params, ~{approx_fp32_mb:.0f} MB in fp32 "
         f"(int8-quantized GGUF/ONNX builds are roughly a quarter of that)")


## Part 5 -- On-Device (Android/Expo) Deployment

**Grounded reality check before you build anything here.** LFM2.5-350M already ships
day-zero on-device support via GGUF (`llama.cpp`), ONNX, and Liquid's own LEAP SDK for
iOS/Android [web:2][web:3][web:13]. Liquid's published on-device numbers for this exact
350M model include, e.g., Snapdragon 8 Elite (Galaxy S25 Ultra) at 15 tok/s decode on NPU
via RunAnywhere, and sub-$300 devices like a Pixel 6a running it at 42 tok/s decode /
328 MB peak memory via the Cactus Engine (int8) [web:14]. Those are real numbers Liquid
measured and published, not something I benchmarked myself.

**The catch for this notebook's custom latent loop**: `llama.cpp`, ONNX Runtime Mobile,
and LEAP all expose token-in/token-out (or a fixed embedding-batch API for
multimodal projectors) -- none of them give you an easy hook to splice a raw PyTorch
hidden-state vector back in as the next input embedding mid-generation the way
`ProductionCoconut` does. Two honest options, not a third "it just works" option:

1. **Keep the latent-reasoning gate server-side.** For Scenarios 1 and 2 this is the
   right call anyway -- they are backend decisions (webhook, tick loop), not on-device
   chat. Ship them as a small FastAPI/ONNX Runtime microservice your Expo app or trading
   backend calls over REST/gRPC. This is what Part 4's benchmark numbers should target.
2. **For genuinely on-device chat/agent features in the Android app**, skip the custom
   latent loop entirely and deploy stock `LFM2.5-350M-GGUF` or `-ONNX` via `llama.cpp` or
   ONNX Runtime Mobile directly [web:2][web:6]. If you need the "reason more before
   answering" behavior on-device, get it by prompting/fine-tuning for a short *visible*
   scratch-pad instead (a few structured tokens the model actually emits), since that is
   natively supported by every mobile runtime, whereas continuous latent thoughts are not.

The cells below implement path 1 end to end (export + serve), since that matches your
existing AVA/trading backend architecture.


In [ ]:
# Export the trained latent-projection heads as small, versioned artifacts.
# These are tiny (hidden_dim x hidden_dim, ~1-4 MB in fp32) -- ship them separately
# from the frozen base model weights, which your service loads once at startup.

import shutil

os.makedirs("artifacts", exist_ok=True)
for name, ckpt_path in [("ava_escalation_gate", CKPT_PATH), ("trading_signal_fusion", TRADING_CKPT)]:
    dest = f"artifacts/{name}.pt"
    shutil.copy(ckpt_path, dest)
    size_kb = os.path.getsize(dest) / 1024
    log.info(f"{name}: {size_kb:.1f} KB -> {dest}")


In [ ]:
# Minimal FastAPI service exposing both gates. Save as service.py and run with
# `uvicorn service:app --host 0.0.0.0 --port 8080` on the backend server that already
# talks to AVA / your trading engine. The Expo app or trading process calls this over
# HTTP, keeping the model out of the mobile binary entirely.

fastapi_service_code = '''
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Coconut Latent Gates")

class AVARequest(BaseModel):
    message_count: int
    sentiment: float
    price_objections: int
    deal_size_tier: int
    negative_flag: int

class TradingRequest(BaseModel):
    rsi_offset: float
    macd_hist: float
    vol_zscore: float

@app.post("/ava/decide")
def ava_decide(req: AVARequest):
    return gate.decide(**req.dict())

@app.post("/trading/fuse")
def trading_fuse(req: TradingRequest):
    return fusion.fuse(**req.dict())

@app.get("/health")
def health():
    return {"status": "ok"}
'''

with open("service.py", "w") as f:
    f.write(fastapi_service_code)
log.info("Wrote service.py -- deploy this next to (or inside) your existing AVA backend")


## Part 6 -- Sanity Tests

Run these before trusting anything above. They are not a substitute for a real held-out evaluation set, but they catch the class of bugs the research notebook had (shape mismatches, non-determinism at temperature=0, cache corruption).

In [ ]:
def test_deterministic_output():
    input_ids = tokenizer.encode("Lead signals: messages=3, sentiment=0.5, price_objections=0, deal_tier=1, negative_flag=0.\nDecision:", return_tensors="pt").to(device)
    r1 = ava_model.generate(input_ids, max_new_tokens=4, temperature=0.0)
    r2 = ava_model.generate(input_ids, max_new_tokens=4, temperature=0.0)
    assert r1["tokens"] == r2["tokens"], "Greedy decode must be deterministic"
    print("PASS: deterministic greedy decode")

def test_no_crash_on_edge_inputs():
    for text in ["Decision:", "Signal:", "x"]:
        ids = tokenizer.encode(text, return_tensors="pt").to(device)
        out = ava_model.generate(ids, max_new_tokens=2)
        assert "text" in out
    print("PASS: no crash on short/edge inputs")

def test_fail_safe_defaults():
    # Simulate a broken checkpoint path scenario at the wrapper level, not the model level.
    broken_result = {"error": "simulated failure"}
    action = "ESCALATE" if broken_result.get("error") else "AUTO"
    assert action == "ESCALATE"
    print("PASS: fail-safe defaults to ESCALATE / HOLD on error")

test_deterministic_output()
test_no_crash_on_edge_inputs()
test_fail_safe_defaults()


## Production Checklist

Before any of the three scenarios touches real users or real capital:

- **AVA escalation gate**: export at least 500-1000 real labeled conversations
  (features + whether a human actually had to step in) and retrain on that instead of the
  synthetic generator; track false-negative rate (missed escalations) as the primary
  metric, not accuracy.
- **Trading signal fusion**: never connect to a live broker until the labels come from a
  proper walk-forward backtest with costs and slippage, and you've paper-traded it for at
  least a few weeks of live-market data.
- **Both gates**: log every `(input, decision, latency)` triple in production so you can
  build the next real training set and detect drift.
- **Both gates**: add a circuit breaker -- if latency p95 exceeds your SLA or the error
  rate spikes, fall back to the old rule-based logic automatically.
- **On-device**: if/when you need genuine on-device reasoning in the Android app, plan for
  a visible short scratch-pad + stock GGUF/ONNX deployment (Part 5), not this latent loop.
- Re-run Part 4's benchmark on the exact hardware you deploy to -- CPU laptop numbers do
  not transfer to your production server or a phone.
